# CircularNet - Waste identification with instance segmentation
Welcome to the Instance Segmentation Notebook! This notebook will take you through the steps of running an  Instance Segmentation model on Images.

## Install Required python packages

In [ ]:
!pip install -q rfdetr==1.5.2 supervision

## Import Libraries

In [ ]:
import json
import os
import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import cv2
from PIL import Image
import supervision as sv
from rfdetr import RFDETRSegMedium

## Load Image Paths

In [ ]:
LIMIT = 20
IMAGE_DIR_PATH = "./images"

all_images = glob.glob(f"{IMAGE_DIR_PATH}/*")[:LIMIT]
print(f"Number of Images : {len(all_images)})

## Load Segmentation Model

In [ ]:
!wget https://storage.googleapis.com/tf_model_garden/vision/waste_identification_ml/CN-ModelCheckpoints/checkpoint_best_total_432x432.pth

In [ ]:
seg_medium_model = RFDETRSegMedium(pretrain_weights="/content/gdrive/MyDrive/ModelInference/checkpoint_best_total_432x432.pth")

## Run the model

In [ ]:
detections = [seg_medium_model.predict(img_path, threshold=0.5) for img_path in tqdm(all_images)]

## Visualize Results

In [ ]:
detections_images = []
for i in np.arange(len(all_images)-1):
    path = all_images[i]
    image = Image.open(path)

    detection = detections[i]

    text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
    thickness = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
    color = sv.ColorPalette.from_hex([
        "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
        "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
    ])

    bbox_annotator = sv.BoxAnnotator(color=color, thickness=thickness)
    mask_annotator = sv.MaskAnnotator()

    label_annotator = sv.LabelAnnotator(
        color=color,
        text_color=sv.Color.BLACK,
        text_scale=text_scale)

    detections_labels = [
        f"{class_id_to_class_name_mapper[int(class_id+1)]} {confidence:.2f}"
        for class_id, confidence
        in zip(detection.class_id, detection.confidence)
    ]

    detections_image = image.copy()
    detections_image = mask_annotator.annotate(detections_image, detection)
    detections_image = bbox_annotator.annotate(detections_image, detection)
    detections_image = label_annotator.annotate(detections_image, detection, detections_labels)

    detections_images.append(detections_image)

# adjust the grid size based upon number of images, below it is set to 5(rows) x 4 (columns) for viewing 20 images
sv.plot_images_grid(images=detections_images, grid_size=(5, 4), size=(20, 20))